In [ ]:
import sys
sys.path.insert(0, '.')
import pandas as pd
from src.parser import parse_deck_list
from src.api_client import enrich_deck
from src.deck import Deck
from src import calculator as calc
from src import monte_carlo as mc


In [ ]:
DECK_LIST = """
Pokémon: 20

4 Abra MEG 54
2 Alakazam ex MEG 57
2 Dunsparce JTG 120
2 Dunsparce TEF 128
1 Psyduck MEP 7
1 Fezandipiti ex SFA 38
1 Shaymin DRI 10
1 Mew ex MEW 151
1 Radiant Greninja ASR 46
1 Ditto MEP 53
1 Cresselia MEP 25
1 Jirachi MEP 126
2 Comfey LOR 79

Trainer: 33

4 Nest Ball SVI 181
4 Ultra Ball SVI 196
3 Iono PAL 269
3 Professor's Research SVI 190
2 Boss's Orders PAL 265
2 Judge SVI 176
2 Lost Vacuum LOR 162
2 Colress's Experiment LOR 155
1 Arven SVI 166
1 Penny SVI 183
1 Rescue Board OBF 159
2 Trekking Shoes ASR 156
1 Counter Catcher PAR 160
2 Mirage Gate LOR 163
1 Sableye LOR 70

Energy: 7

2 Basic Psychic Energy SVE 5
5 Reversal Energy PAL 192
"""


In [ ]:
# Optional: customize target card and searchers for Section 11
TARGET_CARD = {"name": "Alakazam ex", "copies": 2}
SEARCHERS = {
    "Ultra Ball": 4,
    "Nest Ball": 4,
}

# Monte Carlo settings
MC_SIMULATIONS = 100_000
MC_SEED = 42


In [ ]:
print("Parsing deck list...")
parsed = parse_deck_list(DECK_LIST)
print(f"Found {len(parsed)} unique card entries. Looking up via TCGDex API (uses cache)...")
cards = enrich_deck(parsed, cache_path="card_cache.json")
deck = Deck(cards)
print(f"Total cards: {deck.total_cards} | Basic Pokemon: {deck.total_basics}")
unknown = [c for c in deck.cards if c.subcategory == 'unknown']
if unknown:
    print(f"WARNING: {len(unknown)} cards not found in API: {[c.name for c in unknown]}")
else:
    print("All cards classified successfully.")


In [ ]:
print("=" * 50)
print("DECK BREAKDOWN")
print("=" * 50)

breakdown_data = {
    "Category": ["Pokémon", "", "Trainer", "", "", "", "Energy", ""],
    "Subcategory": ["Basic", "Other", "Item", "Supporter", "Stadium", "Tool", "Basic", "Special"],
    "#": [
        sum(c.quantity for c in deck.cards if c.subcategory == "basic"),
        sum(c.quantity for c in deck.cards if c.subcategory == "other"),
        deck.trainers_by_subtype.get("item", 0),
        deck.trainers_by_subtype.get("supporter", 0),
        deck.trainers_by_subtype.get("stadium", 0),
        deck.trainers_by_subtype.get("tool", 0),
        deck.energies_by_subtype.get("basic_energy", 0),
        deck.energies_by_subtype.get("special_energy", 0),
    ]
}
df_breakdown = pd.DataFrame(breakdown_data)
display(df_breakdown)
print(f"Total: {deck.total_cards} cards")


In [ ]:
print("=" * 50)
print("OPENING HAND PROBABILITIES")
print("=" * 50)

N = deck.total_cards
b = deck.total_basics

opening_data = {
    "Event": ["Mulligan (no Basic)", "Starting with exactly 1 Basic", "Starting with 2 or more Basics"],
    "Probability": [
        f"{calc.mulligan_probability(N, b):.2%}",
        f"{calc.exactly_one_basic_probability(N, b):.2%}",
        f"{calc.two_or_more_basics_probability(N, b):.2%}",
    ]
}
display(pd.DataFrame(opening_data))


In [ ]:
print("=" * 50)
print("STARTER PROBABILITIES (per Basic Pokémon)")
print("=" * 50)

starter_data = []
for card in deck.basic_pokemon:
    possible = calc.possible_starter_probability(N, b, card.quantity)
    forced = calc.forced_starter_probability(N, b, card.quantity)
    starter_data.append({
        "Pokémon": f"{card.name} ({card.set_code} #{card.set_number})",
        "Copies": card.quantity,
        "Possible Starter": f"{possible:.2%}",
        "Forced Starter": f"{forced:.2%}",
    })
display(pd.DataFrame(starter_data))


In [ ]:
print("=" * 50)
print("PRIZE CARD PROBABILITIES")
print("=" * 50)

prize_data = []
for card in deck.cards:
    prob = calc.prize_probability(N, card.quantity)
    prize_data.append({
        "Card": card.name,
        "Copies": card.quantity,
        "P(≥1 Prized)": f"{prob:.2%}",
    })
df_prize = pd.DataFrame(prize_data).sort_values("P(≥1 Prized)", ascending=False)
display(df_prize.reset_index(drop=True))


In [ ]:
print("=" * 50)
print("DRAW BY TURN")
print("=" * 50)

MAX_TURNS = 6
draw_data = []
for card in deck.cards:
    if card.quantity == 0:
        continue
    row = {"Card": card.name, "Copies": card.quantity}
    for t in range(1, MAX_TURNS + 1):
        row[f"Turn {t}"] = f"{calc.draw_by_turn_probability(N, card.quantity, t):.2%}"
    draw_data.append(row)
display(pd.DataFrame(draw_data))


In [ ]:
print("=" * 50)
print("SUPPORTER & DEAD HAND")
print("=" * 50)

s = deck.trainers_by_subtype.get("supporter", 0)
e = sum(deck.energies_by_subtype.values())

support_data = {
    "Statistic": [
        "Supporter in opening hand",
        "Dead Hand (0 Supporter + 0 Energy)",
    ],
    "Probability": [
        f"{calc.supporter_turn1_probability(N, s):.2%}",
        f"{calc.dead_hand_probability(N, s, e):.2%}",
    ]
}
display(pd.DataFrame(support_data))


In [ ]:
print("=" * 50)
print("SPECIFIC CARD + SEARCHERS")
print("=" * 50)

target_name = TARGET_CARD["name"]
target_copies = TARGET_CARD["copies"]
s_total = sum(SEARCHERS.values())

p_card = calc.specific_card_in_hand_probability(N, target_copies)
p_searcher = calc.searcher_probability(N, s_total)
p_either = calc.card_or_searcher_probability(N, target_copies, s_total)

searcher_result = {
    "Statistic": [
        f"P({target_name} in opening hand)",
        f"P(Searcher in hand) [{', '.join(SEARCHERS.keys())}]",
        f"P({target_name} OR searcher in hand)",
    ],
    "Probability": [
        f"{p_card:.2%}",
        f"{p_searcher:.2%}",
        f"{p_either:.2%}",
    ]
}
display(pd.DataFrame(searcher_result))


In [ ]:
print("=" * 50)
print(f"MONTE CARLO VALIDATION (N={MC_SIMULATIONS:,} simulations)")
print("=" * 50)

print("Running simulation...")
sim = mc.simulate(deck, n=MC_SIMULATIONS, seed=MC_SEED)

# Compare key stats with theoretical values
comparison_data = [
    {
        "Statistic": "Mulligan rate",
        "Theoretical": f"{calc.mulligan_probability(N, b):.4f}",
        "Simulated": f"{sim['mulligan_rate']:.4f}",
        "Diff": f"{abs(sim['mulligan_rate'] - calc.mulligan_probability(N, b)):.4f}",
    },
    {
        "Statistic": "Supporter in hand",
        "Theoretical": f"{calc.supporter_turn1_probability(N, s):.4f}",
        "Simulated": f"{sim['supporter_in_hand']:.4f}",
        "Diff": f"{abs(sim['supporter_in_hand'] - calc.supporter_turn1_probability(N, s)):.4f}",
    },
    {
        "Statistic": "Dead hand",
        "Theoretical": f"{calc.dead_hand_probability(N, s, e):.4f}",
        "Simulated": f"{sim['dead_hand_rate']:.4f}",
        "Diff": f"{abs(sim['dead_hand_rate'] - calc.dead_hand_probability(N, s, e)):.4f}",
    },
]

# Add possible starters for each basic
for card in deck.basic_pokemon:
    theoretical = calc.possible_starter_probability(N, b, card.quantity)
    simulated = sim["possible_starters"].get(card.name, 0)
    comparison_data.append({
        "Statistic": f"Possible starter: {card.name}",
        "Theoretical": f"{theoretical:.4f}",
        "Simulated": f"{simulated:.4f}",
        "Diff": f"{abs(simulated - theoretical):.4f}",
    })

display(pd.DataFrame(comparison_data))
print("Monte Carlo validation complete.")
